# Reproduction-Ready Time-Series Forecasting Experiment

**CMPE 255 — Assignment 1, Part 2**  
**Methodology:** CRISP-DM

> **Execution disclaimer:** The source dataset is not distributed with this notebook. Consequently, this checked-in notebook contains no executed forecasts, metric values, residual findings, or best-model claim. Supply a dataset as described below and run all cells to generate results.

## 1. Business Understanding

**Objective.** Forecast a single numeric measure at a regular time cadence so that stakeholders can plan capacity, inventory, staffing, or another domain-appropriate resource. Because the source data is unavailable, the exact business variable and forecast horizon must be confirmed with the data owner before operational use.

**Success criteria.** Compare candidate models on an untouched chronological holdout using MAE and RMSE, and MAPE only where actual values are nonzero. Prefer the simplest model whose accuracy, residual behavior, runtime, and maintainability meet stakeholder thresholds. Accuracy thresholds and error costs must be agreed with stakeholders; they are not invented here.

**Risks and constraints.** Structural breaks, calendar effects, sparse history, irregular sampling, missing values, and asymmetric business costs may limit validity. A holdout evaluation estimates historical—not guaranteed future—performance.

## 2. Data Understanding and Reproduction Configuration

Expected input is a CSV with one parseable datetime column and one numeric target. Configuration can be supplied with environment variables. No dataset is downloaded automatically, preserving provenance and access controls.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("default")
plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_SEED = 255
np.random.seed(RANDOM_SEED)

DATA_PATH = Path(os.getenv("TS_DATA_PATH", "data/time_series.csv"))
DATE_COLUMN = os.getenv("TS_DATE_COLUMN")
TARGET_COLUMN = os.getenv("TS_TARGET_COLUMN")
FREQUENCY = os.getenv("TS_FREQUENCY")  # e.g., D, W, MS; inferred when omitted
TEST_SIZE = int(os.getenv("TS_TEST_SIZE", "12"))
SEASONAL_PERIOD = int(os.getenv("TS_SEASONAL_PERIOD", "12"))
MA_WINDOW = int(os.getenv("TS_MA_WINDOW", "3"))
IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)
print({"data_path": str(DATA_PATH), "test_size": TEST_SIZE,
       "seasonal_period": SEASONAL_PERIOD, "moving_average_window": MA_WINDOW})

## 3. Safe Dataset Detection

The next cell deliberately does **not** raise an exception when the dataset is missing. It explains the data contract and sets `RUN_PIPELINE = False`, allowing “Run All” to finish safely. Supply the CSV, configure columns if needed, restart the kernel, and run all cells again.

In [ ]:
RUN_PIPELINE = DATA_PATH.is_file()
raw = None
if not RUN_PIPELINE:
    print(
        f"Dataset not found: {DATA_PATH}\n"
        "Supply a CSV at data/time_series.csv or set TS_DATA_PATH. "
        "The file needs a datetime column and numeric target column. "
        "If names are not recognized, set TS_DATE_COLUMN and TS_TARGET_COLUMN. "
        "Optionally set TS_FREQUENCY, TS_TEST_SIZE, TS_SEASONAL_PERIOD, and TS_MA_WINDOW; "
        "then restart the kernel and Run All. Modeling cells will be skipped safely."
    )
else:
    raw = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns from {DATA_PATH}")

## 4. Datetime Parsing and Initial Data Understanding

Column selection is explicit when environment variables are provided and conservative otherwise. Invalid timestamps and nonnumeric targets are measured before removal; duplicated timestamps are reported and aggregated by mean to create one observation per timestamp.

In [ ]:
series = None
if RUN_PIPELINE:
    date_candidates = [DATE_COLUMN] if DATE_COLUMN else [c for c in raw.columns if c.lower() in {"date", "datetime", "timestamp", "time"}]
    numeric_candidates = [TARGET_COLUMN] if TARGET_COLUMN else [c for c in raw.columns if c not in date_candidates and pd.api.types.is_numeric_dtype(raw[c])]
    if not date_candidates or date_candidates[0] not in raw.columns:
        raise ValueError(f"Set TS_DATE_COLUMN. Available columns: {list(raw.columns)}")
    if not numeric_candidates or numeric_candidates[0] not in raw.columns:
        raise ValueError(f"Set TS_TARGET_COLUMN to a numeric measure. Available columns: {list(raw.columns)}")
    date_col, target_col = date_candidates[0], numeric_candidates[0]
    parsed_dates = pd.to_datetime(raw[date_col], errors="coerce", utc=True).dt.tz_convert(None)
    parsed_values = pd.to_numeric(raw[target_col], errors="coerce")
    print("Invalid timestamps:", int(parsed_dates.isna().sum()))
    print("Missing/non-numeric targets:", int(parsed_values.isna().sum()))
    tidy = pd.DataFrame({"y": parsed_values.to_numpy()}, index=parsed_dates).dropna(axis=0, subset=[])
    tidy = tidy.loc[~tidy.index.isna()].sort_index()
    print("Duplicate timestamp rows:", int(tidy.index.duplicated(keep=False).sum()))
    series = tidy.groupby(level=0)["y"].mean().sort_index()
    print(series.describe())

## 5. Missing-Date Analysis and Regularization

A forecasting cadence must be regular. We infer a frequency when possible, otherwise require `TS_FREQUENCY`. Reindexing exposes missing dates as `NaN`. **No values are filled yet**: doing so before the split could leak information from the test period.

In [ ]:
regular = None
if RUN_PIPELINE:
    inferred_frequency = FREQUENCY or pd.infer_freq(series.index)
    if inferred_frequency is None:
        raise ValueError("Frequency could not be inferred. Set TS_FREQUENCY (for example D, W, or MS).")
    full_index = pd.date_range(series.index.min(), series.index.max(), freq=inferred_frequency)
    missing_dates = full_index.difference(series.index)
    regular = series.reindex(full_index).rename("y")
    print("Frequency:", inferred_frequency)
    print("Missing dates introduced by regularization:", len(missing_dates))
    print("Total missing target values:", int(regular.isna().sum()))
    if len(missing_dates):
        print("First missing dates:", list(missing_dates[:10]))

## 6. Time-Series Exploratory Data Analysis

The plots below examine level, range, possible outliers, and data availability without asserting conclusions before execution.

In [ ]:
if RUN_PIPELINE:
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    regular.plot(ax=axes[0], title="Observed time series (gaps retained)", color="tab:blue")
    axes[0].set_ylabel(target_col)
    axes[1].hist(regular.dropna(), bins=min(30, max(5, int(np.sqrt(regular.notna().sum())))), color="tab:blue", alpha=.8)
    axes[1].set(title="Target distribution", xlabel=target_col, ylabel="Count")
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "eda_overview.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Time-Aware Split, Missing-Value Handling, and Leakage Prevention

The final `TEST_SIZE` timestamps form the holdout. Missing training observations are interpolated using **past-only forward interpolation** and then forward-filled; any leading gap is removed. Test actuals are never imputed for scoring. Forecasts are evaluated only where observed actuals exist.

**Explicit prevention of temporal leakage:**

1. Split chronologically before fitting transformations or models.
2. Never backfill training data from later values.
3. Never use test observations to impute training or recursively create forecasts.
4. Fit every statistical/regression model only on training data.
5. Construct each regression target from later-than-feature timestamps, while keeping all fitted rows within training.
6. Use the holdout once for comparison; use rolling-origin validation for tuning in future work.

In [ ]:
train = test = train_clean = test_observed = None
if RUN_PIPELINE:
    if TEST_SIZE <= 0 or len(regular) <= TEST_SIZE:
        raise ValueError("TS_TEST_SIZE must be positive and smaller than the regularized series.")
    train, test = regular.iloc[:-TEST_SIZE].copy(), regular.iloc[-TEST_SIZE:].copy()
    train_clean = train.interpolate(method="time", limit_direction="forward").ffill().dropna()
    test_observed = test.dropna()
    if train_clean.empty or test_observed.empty:
        raise ValueError("Training data and at least one observed test actual are required.")
    assert train_clean.index.max() < test.index.min(), "Chronological split failed."
    print(f"Train: {train.index.min()} through {train.index.max()} ({len(train)} timestamps)")
    print(f"Test:  {test.index.min()} through {test.index.max()} ({len(test)} timestamps)")
    print("Training missing values after past-only handling:", int(train_clean.isna().sum()))
    print("Observed holdout values available for scoring:", len(test_observed))

## 8. Trend, Seasonality, and Rolling Statistics

Rolling means/standard deviations are descriptive and computed from historical observations. Classical decomposition is attempted only when at least two full seasonal cycles are available; it is a diagnostic, not evidence by itself of stable future seasonality.

In [ ]:
if RUN_PIPELINE:
    rolling_mean = train_clean.rolling(MA_WINDOW, min_periods=1).mean()
    rolling_std = train_clean.rolling(MA_WINDOW, min_periods=2).std()
    fig, ax = plt.subplots(figsize=(12, 5))
    train_clean.plot(ax=ax, label="Training series", alpha=.65)
    rolling_mean.plot(ax=ax, label=f"Rolling mean ({MA_WINDOW})")
    rolling_std.plot(ax=ax, label=f"Rolling std ({MA_WINDOW})")
    ax.set_title("Training-only trend and rolling statistics")
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "rolling_statistics.png", dpi=150, bbox_inches="tight")
    plt.show()

    if len(train_clean) >= 2 * SEASONAL_PERIOD:
        decomposition = seasonal_decompose(train_clean, model="additive", period=SEASONAL_PERIOD, extrapolate_trend="freq")
        decomposition.plot()
        plt.suptitle("Training-only additive trend/seasonality decomposition", y=1.02)
        plt.tight_layout()
        plt.savefig(IMAGES_DIR / "seasonal_decomposition.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Seasonal decomposition skipped: fewer than two complete seasonal cycles.")

## 9. Forecasting Models

All forecasts cover the complete test index without consuming test actuals:

- **Naive:** repeat the last training observation.
- **Moving average:** repeat the mean of the last configured training window (a leakage-safe multi-step forecast).
- **Simple Exponential Smoothing:** level-only exponential smoothing.
- **Holt:** exponential smoothing with an additive trend.
- **Holt-Winters:** additive trend and seasonality; eligible only with at least two full cycles.
- **Optional regression:** direct horizon-specific linear models using lagged training values and deterministic time features. Each horizon is fitted separately, so no test actual enters later-horizon features.

In [ ]:
forecasts = {}
if RUN_PIPELINE:
    horizon = len(test)
    forecasts["Naive"] = pd.Series(train_clean.iloc[-1], index=test.index)
    window = min(MA_WINDOW, len(train_clean))
    forecasts[f"Moving average ({window})"] = pd.Series(train_clean.iloc[-window:].mean(), index=test.index)

    ses_fit = SimpleExpSmoothing(train_clean, initialization_method="estimated").fit(optimized=True)
    forecasts["Simple exponential smoothing"] = pd.Series(ses_fit.forecast(horizon).to_numpy(), index=test.index)

    if len(train_clean) >= 3:
        holt_fit = Holt(train_clean, initialization_method="estimated").fit(optimized=True)
        forecasts["Holt trend"] = pd.Series(holt_fit.forecast(horizon).to_numpy(), index=test.index)

    if len(train_clean) >= 2 * SEASONAL_PERIOD:
        hw_fit = ExponentialSmoothing(
            train_clean, trend="add", seasonal="add", seasonal_periods=SEASONAL_PERIOD,
            initialization_method="estimated"
        ).fit(optimized=True)
        forecasts["Holt-Winters additive"] = pd.Series(hw_fit.forecast(horizon).to_numpy(), index=test.index)
    else:
        print("Holt-Winters skipped: fewer than two complete seasonal cycles.")

### Optional Regression-Based Workflow

Lag count is bounded by the available training history. The feature set includes only lagged targets and deterministic time indices. For horizon `h`, training examples map values known at each origin to the target `h` steps later; predictions use only the final training lags. This direct strategy explicitly prevents recursive use of held-out actuals.

In [ ]:
if RUN_PIPELINE:
    max_lag = min(SEASONAL_PERIOD, max(1, len(train_clean) // 4))
    values = train_clean.to_numpy(dtype=float)
    regression_predictions = []
    for h in range(1, len(test) + 1):
        rows, targets = [], []
        for origin in range(max_lag - 1, len(values) - h):
            lags = values[origin - max_lag + 1:origin + 1][::-1]
            rows.append(np.r_[lags, origin + h])
            targets.append(values[origin + h])
        if len(rows) < max(3, max_lag + 1):
            regression_predictions = []
            print("Regression skipped: insufficient training examples for every direct horizon.")
            break
        model = LinearRegression().fit(np.asarray(rows), np.asarray(targets))
        final_features = np.r_[values[-max_lag:][::-1], len(values) - 1 + h].reshape(1, -1)
        regression_predictions.append(float(model.predict(final_features)[0]))
    if regression_predictions:
        forecasts[f"Direct lag regression ({max_lag} lags)"] = pd.Series(regression_predictions, index=test.index)

## 10. Evaluation: MAE, RMSE, and MAPE

Metrics are computed only at timestamps with observed test actuals. MAE preserves target units, RMSE penalizes large errors, and MAPE provides percentage scale only for nonzero actuals. If every scored actual is zero, MAPE is left as `NaN` rather than misrepresented.

In [ ]:
def safe_mape(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    nonzero = actual != 0
    if not nonzero.any():
        return np.nan
    return np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100

comparison = None
if RUN_PIPELINE:
    rows = []
    for name, prediction in forecasts.items():
        aligned = pd.concat([test.rename("actual"), prediction.rename("forecast")], axis=1).dropna()
        rows.append({
            "model": name,
            "MAE": mean_absolute_error(aligned["actual"], aligned["forecast"]),
            "RMSE": mean_squared_error(aligned["actual"], aligned["forecast"]) ** 0.5,
            "MAPE_percent": safe_mape(aligned["actual"], aligned["forecast"]),
            "n_scored": len(aligned),
        })
    comparison = pd.DataFrame(rows).sort_values(["MAE", "RMSE"], kind="stable").reset_index(drop=True)
    display(comparison)

## 11. Forecast-vs-Actual Visualization and Residual Analysis

Residuals are defined as `actual - forecast`. Plots support—but do not automatically prove—checks for bias, changing variance, outliers, and autocorrelation. Any substantive conclusion requires executing with the real dataset.

In [ ]:
if RUN_PIPELINE:
    fig, ax = plt.subplots(figsize=(12, 6))
    train_clean.iloc[-min(len(train_clean), 3 * TEST_SIZE):].plot(ax=ax, label="Training history", color="black")
    test.plot(ax=ax, label="Actual holdout", color="tab:blue", linewidth=2)
    for name, prediction in forecasts.items():
        prediction.plot(ax=ax, label=name, alpha=.8)
    ax.axvline(test.index.min(), color="gray", linestyle="--", label="Split")
    ax.set_title("Forecasts versus actual holdout")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "forecast_vs_actual.png", dpi=150, bbox_inches="tight")
    plt.show()

    for name, prediction in forecasts.items():
        residuals = (test - prediction).dropna()
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(residuals.index, residuals, marker="o")
        axes[0].axhline(0, color="black", linewidth=1)
        axes[0].set_title(f"Residuals over time — {name}")
        if len(residuals) >= 3:
            plot_acf(residuals, ax=axes[1], lags=min(10, len(residuals) - 1), zero=False)
        else:
            axes[1].text(.5, .5, "Too few residuals for ACF", ha="center", va="center")
            axes[1].set_axis_off()
        fig.tight_layout()
        safe_name = "".join(c if c.isalnum() else "_" for c in name.lower()).strip("_")
        fig.savefig(IMAGES_DIR / f"residuals_{safe_name}.png", dpi=150, bbox_inches="tight")
        plt.show()

## 12. Model Comparison and Final Recommendation Methodology

After execution, apply this auditable decision process:

1. Confirm all candidates were scored on the same observed holdout timestamps.
2. Rank by the stakeholder's primary loss (default: MAE), using RMSE to identify sensitivity to large misses and MAPE only when meaningful.
3. Compare every candidate with the naive baseline; do not deploy a complex model that fails to add material value.
4. Inspect residuals for systematic bias, autocorrelation, changing variance, and event-related errors.
5. Consider runtime, stability, interpretability, data requirements, and operational maintenance.
6. Re-check the finalist with rolling-origin backtesting and business-specific costs before deployment.
7. Recommend a model only after those computed results and stakeholder thresholds are available.

The next cell reports a **provisional metric leader only after execution**. It is not an unexecuted best-model claim.

In [ ]:
if RUN_PIPELINE:
    provisional = comparison.iloc[0]
    print(
        f"Provisional lowest-MAE model on this single holdout: {provisional['model']}. "
        "Treat this as a candidate, not a deployment decision; complete residual review, "
        "rolling-origin validation, and business-cost review first."
    )
else:
    print("No recommendation produced: supply and execute the source dataset first.")

## 13. Deployment Considerations

- Package parsing, frequency enforcement, preprocessing, fit, and forecast steps together and version code, dependencies, data schema, training interval, and model artifact.
- Validate timestamps, duplicates, cadence, missingness, target range, and sufficient history at ingestion; fail safely on schema drift.
- Schedule retraining at a cadence justified by forecast horizon and drift; retain a naive fallback.
- Monitor data freshness, forecast availability, errors once actuals arrive, residual bias, drift, latency, and failure rates.
- Define alert thresholds, owners, rollback procedures, audit logs, access controls, and retraining approval.
- Publish point forecasts with prediction intervals where supported and communicate assumptions to consumers.

## 14. Limitations

- The source dataset, domain semantics, sampling cadence, and business loss function were unavailable and therefore not validated here.
- A single holdout can be sensitive to the selected period; it is not a substitute for rolling-origin backtesting.
- Classical exponential-smoothing assumptions may fail under multiple seasonalities, external drivers, abrupt breaks, nonlinear behavior, or intermittent demand.
- Interpolation can distort genuine shocks; the missingness mechanism needs domain review.
- The optional regression uses basic lags and time trend only; no unavailable exogenous variables are assumed.
- Prediction intervals, hyperparameter tuning, and statistical significance comparisons are outside this template.

## 15. Future Improvements

1. Add expanding-window rolling-origin cross-validation and tune windows, trend damping, seasonal form, and lag sets without touching the final holdout.
2. Incorporate approved calendar, event, weather, price, or other known-in-advance exogenous features.
3. Test transformations, robust outlier handling, multiple seasonalities, intermittent-demand models, and probabilistic forecasts.
4. Evaluate interval coverage and business-weighted/asymmetric error measures.
5. Add automated data validation, unit tests, experiment tracking, drift monitoring, and scheduled retraining.

## 16. CRISP-DM Conclusion

This notebook maps business objectives to data understanding, leakage-safe preparation, baseline and classical/statistical modeling, holdout evaluation, and deployment planning. It is intentionally reproduction-ready rather than result-bearing: because the protected source dataset was not supplied or run, no forecasts, numerical metrics, residual conclusions, or best-model assertion are reported. Once authorized data and stakeholder criteria are available, execute the workflow, validate it with rolling-origin evaluation, and document the evidence supporting the final recommendation.